<a href="https://colab.research.google.com/github/EthanIsaac/valorant-voiceline-qa/blob/main/notebooks/01_agent_voice_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Agent Voice Classifier: fine-tuning 3 audio models

**Project:** VALORANT Voice-Line QA & Localization Assistant (Riot Games)
**Pipeline being fine-tuned:** `audio-classification`, which identifies **which of the 26 agents** is speaking in a voice-line recording.

**Why it matters:** each agent ships with hundreds of voice lines, recorded and localized into many languages. Audio QA must confirm that every file contains the right agent's voice before it reaches the game.

**Notebook outline**
- Step 0: Setup (packages, seed, output folder)
- Step 1: Load the dataset at a pinned revision
- Step 2: Clean the data and split it into train / validation / test
- Save and download the results (with SHA-256 fingerprints)

All results are written to `results/agent_classifier/` so the report can quote them exactly.

> Runtime: **GPU** (Runtime → Change runtime type → T4 GPU).

# Step 0: Setup

In [ ]:
# Same transformers constraint as the course notebooks
# (pandas, scikit-learn, soundfile, librosa and huggingface_hub are already installed in Colab).
# Expected: a pip "dependency conflicts" message about diffusers/gradio needing a newer
# huggingface-hub. Those packages are not used here, so it can be ignored.
%pip install -q "transformers>=4.46.0,<5.0.0" accelerate

In [ ]:
import io
import json
import os
import re
import time

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from transformers import set_seed

# One seed for everything (Python, NumPy, PyTorch), so reruns give the same results
SEED = 42
set_seed(SEED)

# Every result file produced by this notebook goes in this folder
RESULTS_DIR = "results/agent_classifier"
os.makedirs(RESULTS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Step 1: Load the dataset

**Source:** [NgThVinh/ValorantAgentVoiceLines](https://huggingface.co/datasets/NgThVinh/ValorantAgentVoiceLines), in-game voice lines of every VALORANT agent with their transcripts (collected from the VALORANT Fandom wiki).
Each agent is stored as a separate file. The **agent name is the label** we want to predict.

We load a **fixed revision** (commit hash) so the data can never change under us. The download is about 2 GB and takes a couple of minutes.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

DATASET = "NgThVinh/ValorantAgentVoiceLines"
REVISION = "c2e0d3493f44da5be07090d6ebb9b69ebbcb2060"   # pinned commit

# One parquet file per agent: "<agent>/train-00000-of-00001.parquet"
parquet_files = sorted(
    f for f in list_repo_files(DATASET, repo_type="dataset", revision=REVISION) if f.endswith(".parquet")
)

frames = []
for file in parquet_files:
    path = hf_hub_download(DATASET, file, repo_type="dataset", revision=REVISION)
    agent_df = pd.read_parquet(path)
    agent_df["agent"] = file.split("/")[0]
    frames.append(agent_df)

data = pd.concat(frames, ignore_index=True)
data["audio_bytes"] = data["audio_file"].map(lambda audio: audio["bytes"])   # raw WAV file content
data = data.drop(columns="audio_file")

AGENTS = sorted(data["agent"].unique())
print(f"{len(data)} voice lines from {len(AGENTS)} agents")
print(data.groupby("agent").size().to_string())

In [ ]:
# Listen to a few examples
for _, row in data.sample(3, random_state=SEED).iterrows():
    print(f"{row['agent']:<10} {row['audio_name']:<28} {row['transcript']}")
    display(Audio(row["audio_bytes"]))

# Step 2: Clean the data and split it

**Audio checks.** Every clip is a mono WAV recorded at 48 kHz. Clips shorter than **0.5 s** (e.g. "Mm-hmm") carry too little voice to identify a speaker, so they are removed.

**Transcript cleaning.** The transcripts contain wiki markup. The words actually spoken are always the parts **inside double quotes**:

| Raw transcript | Spoken text |
|---|---|
| `"Hey hermanos, spend everything you have." (''Hermanos'': Siblings)` | `Hey hermanos, spend everything you have.` (translator note removed) |
| `''sighs'' "Cool"` | `Cool` (stage direction removed) |
| `"Sure, but if anything explodes again, you are telling Brimstone\|Brim."` | `... you are telling Brim.` (wiki link: the second name is the one spoken) |
| `''Jett groans''` | *(no speech)* |

Two flags are added for the speech-to-text notebook: `has_speech` (the clip contains words) and `english_only` (no foreign words or translator notes).
Non-speech clips (groans, sighs) are **kept** for agent identification, because they are still the agent's voice.

**Train / validation / test split (80 / 10 / 10).** Many lines were recorded several times (e.g. two takes of *"Keep up."* by the same agent).
If one take were in training and another in test, the test would be too easy (**leakage**). So the split is made **per agent and by line**:
all takes of the same line by the same agent go to the same split. Every agent appears in all three splits.

In [ ]:
MIN_DURATION_S = 0.5
SPLIT_FRACTIONS = {"train": 0.8, "validation": 0.1, "test": 0.1}


def spoken_text(transcript):
    """Keep only the quoted (spoken) parts; in wiki links 'Name|Nickname' keep the spoken nickname."""
    quoted = re.findall(r'"([^"]*)"', str(transcript))
    text = " ".join(part.strip() for part in quoted)
    text = re.sub(r"[^\s|]+\|([^\s,.!?]+)", r"\1", text)
    return " ".join(text.split())


data["duration_s"] = data["audio_bytes"].map(lambda b: sf.info(io.BytesIO(b)).duration)
data["spoken_text"] = data["transcript"].map(spoken_text)
data["has_speech"] = data["spoken_text"] != ""
data["english_only"] = (
    data["has_speech"]
    & ~data["transcript"].str.contains(r"\(", regex=True)                       # no translator note
    & ~data["spoken_text"].str.contains(r"[^\x00-\x7F’‘“”…—–]", regex=True)   # no non-English characters
)

cleaning_log = [("original", len(data))]
data = data[data["duration_s"] >= MIN_DURATION_S].reset_index(drop=True)
cleaning_log.append((f"removed clips shorter than {MIN_DURATION_S}s", len(data)))

print(pd.DataFrame(cleaning_log, columns=["step", "clips"]).to_string(index=False))
print(f"\nDuration (s): median {data['duration_s'].median():.2f}, 95th pct {data['duration_s'].quantile(0.95):.2f}, max {data['duration_s'].max():.2f}")
print(f"Clips with speech: {data['has_speech'].sum()} | English-only: {data['english_only'].sum()}")

# Before/after examples of the transcript cleaning
data.loc[data["transcript"].str.contains(r"\(|''|\|", regex=True), ["transcript", "spoken_text"]].sample(8, random_state=SEED)

In [ ]:
# Group = one line of one agent (all takes of that line). Non-speech clips are grouped by their raw description.
line = np.where(data["has_speech"], data["spoken_text"], data["transcript"]).astype(str)
data["line_group"] = data["agent"] + "::" + pd.Series(line).str.lower().str.strip()

rng = np.random.default_rng(SEED)
group_split = {}
for agent, groups in data.groupby("agent")["line_group"]:
    groups = sorted(groups.unique())
    rng.shuffle(groups)
    n_test = round(len(groups) * SPLIT_FRACTIONS["test"])
    n_val = round(len(groups) * SPLIT_FRACTIONS["validation"])
    for i, group in enumerate(groups):
        group_split[group] = "test" if i < n_test else "validation" if i < n_test + n_val else "train"

data["split"] = data["line_group"].map(group_split)

# Leakage check: every line group belongs to exactly one split
assert data.groupby("line_group")["split"].nunique().max() == 1

summary = data.groupby("split").agg(
    clips=("audio_name", "size"),
    agents=("agent", "nunique"),
    hours=("duration_s", lambda d: round(d.sum() / 3600, 2)),
    english_only=("english_only", "sum"),
).loc[["train", "validation", "test"]]
summary

In [ ]:
# Save the split manifest (used by every later notebook) and the cleaning record
manifest_cols = ["agent", "audio_name", "split", "duration_s", "transcript", "spoken_text", "has_speech", "english_only"]
data[manifest_cols].sort_values(["agent", "audio_name"]).to_csv(f"{RESULTS_DIR}/splits.csv", index=False)

with open(f"{RESULTS_DIR}/data_cleaning.json", "w") as f:
    json.dump({
        "dataset": DATASET,
        "revision": REVISION,
        "agents": AGENTS,
        "min_duration_s": MIN_DURATION_S,
        "cleaning_steps": cleaning_log,
        "split_sizes": summary.reset_index().to_dict(orient="records"),
        "clips_per_agent_per_split": data.groupby(["agent", "split"]).size().unstack().to_dict(orient="index"),
    }, f, indent=2)

print(os.listdir(RESULTS_DIR))

# Save and download the results

Every file this notebook generated is listed with its **SHA-256 fingerprint** in `SHA256SUMS`.
Rerunning the notebook must reproduce the same fingerprints for the deterministic files (data split and cleaning record).
Then everything is zipped and downloaded as `results_agent_classifier.zip`.

In [ ]:
import hashlib
from pathlib import Path

result_files = sorted(p for p in Path(RESULTS_DIR).rglob("*") if p.is_file() and p.name != "SHA256SUMS")
with open(f"{RESULTS_DIR}/SHA256SUMS", "w") as f:
    for p in result_files:
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        f.write(f"{digest}  {p.relative_to(RESULTS_DIR)}\n")
        print(f"{digest[:16]}…  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size:,} bytes)")

In [ ]:
!rm -f results_agent_classifier.zip && zip -qrX results_agent_classifier.zip results
from google.colab import files
files.download("results_agent_classifier.zip")